# 01. Data Loading and Cleaning Pipeline
## Nepal Terai Extreme Heatwave & Climate Dataset (1981–2026)

This notebook loads the raw NASA POWER satellite meteorological dataset for 21 Terai districts, cleans it, handles missing/fill values, and prepares it for Exploratory Data Analysis (EDA) and Machine Learning.


In [125]:
import os
import pandas as pd
import numpy as np


###  Load Raw Dataset

In [126]:
raw_data_path = "../data/raw/terai_heatwave_raw.csv"
df = pd.read_csv(raw_data_path)
df.head()

,YEAR,DOY,Province,District,Regional_Hub,Latitude,Longitude,T2M,T2M_MAX,T2M_MIN,RH2M,WS2M,PRECTOTCORR
0,2026,264,Bagmati,Chitwan,Bharatpur,27.68,84.43,NaN,NaN,NaN,NaN,NaN,NaN
1,2026,264,Gandaki,Nawalpur,Kawasoti,27.65,84.12,NaN,NaN,NaN,NaN,NaN,NaN
2,2026,264,Koshi,Jhapa,Bhadrapur,26.54,88.08,NaN,NaN,NaN,NaN,NaN,NaN
3,2026,264,Koshi,Morang,Biratnagar,26.45,87.27,NaN,NaN,NaN,NaN,NaN,NaN
4,2026,264,Koshi,Sunsari,Inaruwa,26.60,87.15,NaN,NaN,NaN,NaN,NaN,NaN


In [127]:
df.shape

(204981, 13)

### Rename columns

In [128]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 204981 entries, 0 to 204980
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   YEAR          204981 non-null  int64  
 1   DOY           204981 non-null  int64  
 2   Province      204981 non-null  str    
 3   District      204981 non-null  str    
 4   Regional_Hub  204981 non-null  str    
 5   Latitude      204981 non-null  float64
 6   Longitude     204981 non-null  float64
 7   T2M           204897 non-null  float64
 8   T2M_MAX       204897 non-null  float64
 9   T2M_MIN       204897 non-null  float64
 10  RH2M          204897 non-null  float64
 11  WS2M          204897 non-null  float64
 12  PRECTOTCORR   204897 non-null  float64
dtypes: float64(8), int64(2), str(3)
memory usage: 24.7 MB


In [129]:
column_mapping = {
    "YEAR": "Year",
    "DOY": "Day_of_Year",
    "T2M": "Mean_Temp_C",
    "T2M_MAX": "Max_Temp_C",
    "T2M_MIN": "Min_Temp_C",
    "RH2M": "Humidity_Pct",
    "WS2M": "Wind_Speed_ms",
    "PRECTOTCORR": "Precipitation_mm"
}

df = df.rename(columns=column_mapping)
df.head()


,Year,Day_of_Year,Province,District,Regional_Hub,Latitude,Longitude,Mean_Temp_C,Max_Temp_C,Min_Temp_C,Humidity_Pct,Wind_Speed_ms,Precipitation_mm
0,2026,264,Bagmati,Chitwan,Bharatpur,27.68,84.43,NaN,NaN,NaN,NaN,NaN,NaN
1,2026,264,Gandaki,Nawalpur,Kawasoti,27.65,84.12,NaN,NaN,NaN,NaN,NaN,NaN
2,2026,264,Koshi,Jhapa,Bhadrapur,26.54,88.08,NaN,NaN,NaN,NaN,NaN,NaN
3,2026,264,Koshi,Morang,Biratnagar,26.45,87.27,NaN,NaN,NaN,NaN,NaN,NaN
4,2026,264,Koshi,Sunsari,Inaruwa,26.60,87.15,NaN,NaN,NaN,NaN,NaN,NaN


### Clean String Data

In [130]:
for col in ["Province", "District", "Regional_Hub"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title()

### Construct Calendar Date



In [131]:
df["Date"] = pd.to_datetime(
        df["Year"].astype(str) + "-" + df["Day_of_Year"].astype(str).str.zfill(3),
        format="%Y-%j"
    )
df["Date"].sample(5)


14056    2024-11-21
133714   2009-04-16
107846   2012-08-30
18114    2024-05-12
10       2026-09-21
Name: Date, dtype: datetime64[us]

### 4. convert datatypes



In [132]:


category_cols = ["District", "Province", "Regional_Hub"]
for col in category_cols:
    df[col] = df[col].astype("category")

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 204981 entries, 0 to 204980
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Year              204981 non-null  int64         
 1   Day_of_Year       204981 non-null  int64         
 2   Province          204981 non-null  category      
 3   District          204981 non-null  category      
 4   Regional_Hub      204981 non-null  category      
 5   Latitude          204981 non-null  float64       
 6   Longitude         204981 non-null  float64       
 7   Mean_Temp_C       204897 non-null  float64       
 8   Max_Temp_C        204897 non-null  float64       
 9   Min_Temp_C        204897 non-null  float64       
 10  Humidity_Pct      204897 non-null  float64       
 11  Wind_Speed_ms     204897 non-null  float64       
 12  Precipitation_mm  204897 non-null  float64       
 13  Date              204981 non-null  datetime64[us]
dtypes: category(3),

### Remove Duplicate Records



In [133]:
duplicates = df.duplicated(subset=["Date", "District"]).sum()
print(f"Duplicate rows found: {duplicates}")
df = df.drop_duplicates(subset=["Date", "District"]).reset_index(drop=True)

Duplicate rows found: 0


### Remove Empty Data

In [134]:
df.isna().sum()

Year                 0
Day_of_Year          0
Province             0
District             0
Regional_Hub         0
Latitude             0
Longitude            0
Mean_Temp_C         84
Max_Temp_C          84
Min_Temp_C          84
Humidity_Pct        84
Wind_Speed_ms       84
Precipitation_mm    84
Date                 0
dtype: int64

In [135]:
df.isna().mean()*100

Year                0.000000
Day_of_Year         0.000000
Province            0.000000
District            0.000000
Regional_Hub        0.000000
Latitude            0.000000
Longitude           0.000000
Mean_Temp_C         0.040979
Max_Temp_C          0.040979
Min_Temp_C          0.040979
Humidity_Pct        0.040979
Wind_Speed_ms       0.040979
Precipitation_mm    0.040979
Date                0.000000
dtype: float64

In [136]:
df = df.dropna().reset_index(drop=True)


In [137]:
df.isna().sum()


Year                0
Day_of_Year         0
Province            0
District            0
Regional_Hub        0
Latitude            0
Longitude           0
Mean_Temp_C         0
Max_Temp_C          0
Min_Temp_C          0
Humidity_Pct        0
Wind_Speed_ms       0
Precipitation_mm    0
Date                0
dtype: int64

### Meteorological Sanity Checks


In [138]:
df = df[df["Min_Temp_C"] <= df["Max_Temp_C"]]
df = df[(df["Humidity_Pct"] >= 0.0) & (df["Humidity_Pct"] <= 100.0)]
df = df[(df["Wind_Speed_ms"] >= 0.0) & (df["Precipitation_mm"] >= 0.0)]

### Order Columns & Sort Chronologically


In [139]:

df = df.sort_values(
    by=["Date", "Province", "Regional_Hub", "District"],
    ascending=[False, True, True, True]
).reset_index(drop=True)

ordered_cols = [
    "Date", "Year", "Day_of_Year", "Province", "District", "Regional_Hub",
    "Latitude", "Longitude", "Max_Temp_C", "Min_Temp_C", "Mean_Temp_C",
    "Humidity_Pct", "Wind_Speed_ms", "Precipitation_mm"
]

clean_df = df[ordered_cols]

clean_df.head()

,Date,Year,Day_of_Year,Province,District,Regional_Hub,Latitude,Longitude,Max_Temp_C,Min_Temp_C,Mean_Temp_C,Humidity_Pct,Wind_Speed_ms,Precipitation_mm
0,2026-09-17,2026,260,Bagmati,Chitwan,Bharatpur,27.68,84.43,29.43,24.65,26.43,91.49,1.15,12.56
1,2026-09-17,2026,260,Gandaki,Nawalpur,Kawasoti,27.65,84.12,29.43,24.65,26.43,91.49,1.15,12.56
2,2026-09-17,2026,260,Koshi,Jhapa,Bhadrapur,26.54,88.08,30.02,24.77,26.82,90.43,0.92,9.43
3,2026-09-17,2026,260,Koshi,Morang,Biratnagar,26.45,87.27,30.55,25.11,27.26,89.39,0.73,8.12
4,2026-09-17,2026,260,Koshi,Sunsari,Inaruwa,26.60,87.15,31.03,25.35,27.71,88.39,0.78,7.52


### Summary Statistics


In [140]:
clean_df.describe().round(2)

,Date,Year,Day_of_Year,Latitude,Longitude,Max_Temp_C,Min_Temp_C,Mean_Temp_C,Humidity_Pct,Wind_Speed_ms,Precipitation_mm
count,204897,204897.00,204897.00,204897.00,204897.00,204897.00,204897.00,204897.00,204897.00,204897.00,204897.00
mean,2013-05-09 23:59:59.999999,2012.86,181.73,27.32,84.42,30.88,19.92,24.87,57.59,1.68,3.68
min,2000-01-01 00:00:00,2000.00,1.00,26.45,80.18,11.77,0.07,6.63,4.36,0.37,0.00
25%,2006-09-05 00:00:00,2006.00,91.00,26.65,83.05,26.79,14.52,19.81,39.92,1.18,0.00
50%,2013-05-10 00:00:00,2013.00,181.00,27.03,84.87,30.77,21.68,26.41,57.43,1.51,0.08
75%,2020-01-13 00:00:00,2020.00,272.00,27.68,85.92,34.71,25.21,29.21,78.83,1.99,3.14
max,2026-09-17 00:00:00,2026.00,366.00,28.97,88.08,48.80,34.71,41.32,97.03,7.51,272.13
std,NaN,7.71,105.09,0.73,2.20,5.63,6.15,5.98,22.22,0.70,9.03
